# Product Performance

This notebook ranks products by (1) total recorded quantity and (2) the number of distinct invoices containing the product. It uses rows with `Quantity > 0` and `UnitPrice > 0` as a **provisional sales-like population**. This sign-based rule is not a verified business definition of a completed sale.

Descriptions are audited as labels, not used as product keys. All product metrics are grouped by `StockCode`; normalized descriptions are joined afterward so inconsistent labels cannot split or inflate product totals.

## 1. Load and preserve the source

Load the original workbook into `retail`, validate the expected columns, and keep a snapshot to check that later analysis leaves the loaded data unchanged.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_path = project_root / "data" / "raw" / "Online Retail.xlsx"
retail = pd.read_excel(data_path, engine="openpyxl")

required_columns = {"InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice"}
missing_columns = required_columns.difference(retail.columns)
assert not missing_columns, f"Required columns are missing: {sorted(missing_columns)}"
source_snapshot = retail.copy(deep=True)

print(f"Workbook: {data_path}")
print(f"Source rows: {len(retail):,}; columns: {retail.shape[1]}")
print(f"Missing StockCode: {retail['StockCode'].isna().sum():,}; missing InvoiceNo: {retail['InvoiceNo'].isna().sum():,}")

Workbook: c:\Users\pc\Desktop\omar benamara\OMAR Bureau\pricing-demand-analysis\data\raw\Online Retail.xlsx
Source rows: 541,909; columns: 8
Missing StockCode: 0; missing InvoiceNo: 0


## 2. Define the provisional population

Apply the requested positive-quantity, positive-price rule to a separate copy. The checks make population coverage explicit and catch changes in the input workbook.

In [2]:
sales_like_mask = retail["Quantity"].gt(0) & retail["UnitPrice"].gt(0)
sales_like = retail.loc[sales_like_mask].copy()

population_summary = {
    "rows": len(sales_like),
    "distinct_stock_codes": sales_like["StockCode"].nunique(),
    "distinct_invoices": sales_like["InvoiceNo"].nunique(),
    "total_quantity": sales_like["Quantity"].sum(),
    "missing_descriptions": sales_like["Description"].isna().sum(),
}
display(pd.Series(population_summary, name="value"))

assert sales_like["Quantity"].gt(0).all()
assert sales_like["UnitPrice"].gt(0).all()
assert sales_like["StockCode"].notna().all(), "Product grouping requires non-missing StockCode values."
assert sales_like["InvoiceNo"].notna().all(), "Invoice counts require non-missing InvoiceNo values."

rows                     530104
distinct_stock_codes       3922
distinct_invoices         19960
total_quantity          5588376
missing_descriptions          0
Name: value, dtype: int64

## 3. Aggregate product performance

Aggregate only by `StockCode`. Quantity is summed across qualifying rows; invoice breadth is the number of distinct qualifying `InvoiceNo` values for each stock code.

In [3]:
product_metrics = (
    sales_like.groupby("StockCode", as_index=False, dropna=False)
    .agg(
        total_quantity=("Quantity", "sum"),
        distinct_invoices=("InvoiceNo", "nunique"),
        qualifying_rows=("StockCode", "size"),
    )
)

print(f"Products in aggregated table: {len(product_metrics):,}")
print(f"Grouped quantity total: {product_metrics['total_quantity'].sum():,}")
print(f"Grouped qualifying rows: {product_metrics['qualifying_rows'].sum():,}")

Products in aggregated table: 3,922
Grouped quantity total: 5,588,376
Grouped qualifying rows: 530,104


## 4. Audit descriptions separately

For auditing only, trim whitespace, collapse repeated spaces, and case-fold descriptions. Count normalized variants per stock code and choose a display label only when its normalized form has a unique highest row count. A tie is left unlabeled as ambiguous; missing descriptions remain visible in the audit. The source descriptions are not overwritten.

In [4]:
description_rows = sales_like[["StockCode", "Description"]].copy()
description_rows["description_normalized"] = (
    description_rows["Description"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
)
description_rows["description_normalized"] = description_rows["description_normalized"].replace("", pd.NA)

description_variants = (
    description_rows.dropna(subset=["description_normalized"])
    .groupby(["StockCode", "description_normalized"]).size()
    .rename("description_rows").reset_index()
)
variant_stats = description_variants.groupby("StockCode").agg(
    description_variant_count=("description_normalized", "nunique"),
    described_rows=("description_rows", "sum"),
)
missing_description_counts = sales_like.groupby("StockCode")["Description"].agg(
    description_missing_rows=lambda values: values.isna().sum()
)

if not description_variants.empty:
    top_variant_count = description_variants.groupby("StockCode")["description_rows"].transform("max")
    top_variants = description_variants.loc[description_variants["description_rows"].eq(top_variant_count)]
    top_variant_counts = top_variants.groupby("StockCode").size()
    unique_leader_codes = top_variant_counts.loc[top_variant_counts.eq(1)].index
    canonical_labels = (
        top_variants.loc[top_variants["StockCode"].isin(unique_leader_codes), ["StockCode", "description_normalized"]]
        .set_index("StockCode")["description_normalized"]
        .rename("canonical_description")
    )
else:
    canonical_labels = pd.Series(dtype="string", name="canonical_description")

description_audit = pd.DataFrame(index=pd.Index(sales_like["StockCode"].unique(), name="StockCode"))
description_audit = description_audit.join(variant_stats).join(missing_description_counts).join(canonical_labels)
description_audit["description_variant_count"] = description_audit["description_variant_count"].fillna(0).astype(int)
description_audit["described_rows"] = description_audit["described_rows"].fillna(0).astype(int)
description_audit["description_missing_rows"] = description_audit["description_missing_rows"].fillna(0).astype(int)
description_audit["description_audit_status"] = "ambiguous_top_label"
description_audit.loc[description_audit["description_variant_count"].eq(0), "description_audit_status"] = "missing_description"
description_audit.loc[description_audit["description_variant_count"].eq(1), "description_audit_status"] = "single_normalized_label"
description_audit.loc[description_audit["canonical_description"].notna() & description_audit["description_variant_count"].gt(1), "description_audit_status"] = "unique_most_frequent_label"

print("Product counts by description-audit status:")
display(description_audit["description_audit_status"].value_counts().rename_axis("status").to_frame("products"))
print(f"Qualifying rows with missing Description: {sales_like['Description'].isna().sum():,}")
print("Examples of products with multiple normalized variants or unresolved labels:")
audit_examples = description_audit.loc[
    description_audit["description_variant_count"].gt(1) | description_audit["description_audit_status"].eq("missing_description")
].head(10)
display(audit_examples.reset_index())
if not audit_examples.empty:
    display(description_variants.loc[description_variants["StockCode"].isin(audit_examples.index)].sort_values(["StockCode", "description_rows", "description_normalized"], ascending=[True, False, True]).head(30))

Product counts by description-audit status:


,products
status,
single_normalized_label,3710
unique_most_frequent_label,210
ambiguous_top_label,2


Qualifying rows with missing Description: 0
Examples of products with multiple normalized variants or unresolved labels:


,StockCode,description_variant_count,described_rows,description_missing_rows,canonical_description,description_audit_status
0,85123A,2,2265,0,white hanging heart t-light holder,unique_most_frequent_label
1,71053,2,346,0,white metal lantern,unique_most_frequent_label
2,22632,2,405,0,hand warmer red retrospot,unique_most_frequent_label
3,82486,2,616,0,wood s/3 cabinet ant white finish,unique_most_frequent_label
4,84997B,2,374,0,childrens cutlery retrospot red,unique_most_frequent_label
5,84997C,2,336,0,childrens cutlery polkadot blue,unique_most_frequent_label
6,20725,2,1595,0,lunch bag red retrospot,unique_most_frequent_label
7,21175,2,818,0,gin + tonic diet metal sign,unique_most_frequent_label
8,22778,2,148,0,glass cloche small,unique_most_frequent_label
9,22466,2,212,0,fairy tale cottage night light,unique_most_frequent_label


,StockCode,description_normalized,description_rows
101,20725,lunch bag red retrospot,1594
102,20725,lunch bag red spotty,1
342,21175,gin + tonic diet metal sign,768
343,21175,gin and tonic diet metal sign,50
1300,22466,fairy tale cottage night light,117
1301,22466,fairy tale cottage nightlight,95
1465,22632,hand warmer red retrospot,387
1464,22632,hand warmer red polka dot,18
1611,22778,glass cloche small,147
1610,22778,glass bell jar small,1


## 5. Rank products on each measure

Attach the description audit after aggregation, then show separate top-20 tables. Ties are resolved by ascending `StockCode` for reproducible display.

In [5]:
product_performance = product_metrics.merge(
    description_audit.reset_index(), on="StockCode", how="left", validate="one_to_one"
)

top_by_quantity = product_performance.sort_values(
    ["total_quantity", "StockCode"], ascending=[False, True], kind="mergesort"
).head(20)
top_by_invoices = product_performance.sort_values(
    ["distinct_invoices", "StockCode"], ascending=[False, True], kind="mergesort"
).head(20)

result_columns = [
    "StockCode", "canonical_description", "description_audit_status",
    "total_quantity", "distinct_invoices", "qualifying_rows",
]
print("Top 20 products by total recorded quantity:")
display(top_by_quantity[result_columns].reset_index(drop=True))
print("Top 20 products by distinct qualifying invoices:")
display(top_by_invoices[result_columns].reset_index(drop=True))

Top 20 products by total recorded quantity:


,StockCode,canonical_description,description_audit_status,total_quantity,distinct_invoices,qualifying_rows
0,23843,"paper craft , little birdie",single_normalized_label,80995,1,1
1,23166,medium ceramic top storage jar,single_normalized_label,78033,247,250
2,22197,popcorn holder,unique_most_frequent_label,56921,1392,1426
3,84077,world war 2 gliders asstd designs,single_normalized_label,55047,535,537
4,85099B,jumbo bag red retrospot,single_normalized_label,48474,2089,2112
5,85123A,white hanging heart t-light holder,unique_most_frequent_label,37660,2198,2265
6,84879,assorted colour bird ornament,single_normalized_label,36461,1455,1489
7,21212,pack of 72 retrospot cake cases,single_normalized_label,36419,1320,1370
8,23084,rabbit night light,single_normalized_label,30788,994,1036
9,22492,mini paint set vintage,single_normalized_label,26633,380,380


Top 20 products by distinct qualifying invoices:


,StockCode,canonical_description,description_audit_status,total_quantity,distinct_invoices,qualifying_rows
0,85123A,white hanging heart t-light holder,unique_most_frequent_label,37660,2198,2265
1,85099B,jumbo bag red retrospot,single_normalized_label,48474,2089,2112
2,22423,regency cakestand 3 tier,single_normalized_label,13879,1988,2017
3,47566,party bunting,single_normalized_label,18295,1685,1706
4,20725,lunch bag red retrospot,unique_most_frequent_label,19553,1565,1595
5,84879,assorted colour bird ornament,single_normalized_label,36461,1455,1489
6,22197,popcorn holder,unique_most_frequent_label,56921,1392,1426
7,22720,set of 3 cake tins pantry design,single_normalized_label,7493,1385,1399
8,21212,pack of 72 retrospot cake cases,single_normalized_label,36419,1320,1370
9,22383,lunch bag suki design,single_normalized_label,12552,1284,1326


## 6. Validate aggregation and audit joins

Reconcile grouped rows and units to the filtered source population, confirm one row per product, bound invoice counts, and check that adding descriptions did not change product coverage. These checks are assertions: a mismatch should stop execution for investigation.

In [6]:
population_invoice_count = sales_like["InvoiceNo"].nunique()

assert retail.equals(source_snapshot), "The loaded source dataframe was modified during analysis."
assert product_metrics["StockCode"].is_unique, "The aggregated table has duplicate StockCode keys."
assert len(product_metrics) == sales_like["StockCode"].nunique(), "Product count does not reconcile."
assert product_metrics["total_quantity"].sum() == sales_like["Quantity"].sum(), "Grouped quantity does not reconcile."
assert product_metrics["qualifying_rows"].sum() == len(sales_like), "Grouped row counts do not reconcile."
assert product_metrics["distinct_invoices"].between(1, population_invoice_count).all(), "A product invoice count is outside valid population bounds."
assert len(product_performance) == len(product_metrics), "Description join changed product row count."
assert product_performance["StockCode"].is_unique, "Description join duplicated product keys."
assert product_performance["description_variant_count"].notna().all(), "Description audit is missing product coverage."
assert product_performance["description_missing_rows"].sum() == sales_like["Description"].isna().sum(), "Missing-description audit does not reconcile."

expected_population = {"rows": 530104, "distinct_stock_codes": 3922, "distinct_invoices": 19960, "total_quantity": 5588376}
for metric, expected_value in expected_population.items():
    actual_value = population_summary[metric]
    assert actual_value == expected_value, f"Unexpected {metric}: got {actual_value:,}, expected {expected_value:,}; inspect the source workbook and filter."

print("PASS: source frame unchanged")
print("PASS: qualifying population satisfies Quantity > 0 and UnitPrice > 0")
print("PASS: product keys, grouped rows, total quantity, and invoice bounds reconcile")
print("PASS: description audit reconciles and its join preserves product rows")
print("PASS: population matches the previously documented workbook counts")
print(f"Missing Description rows retained in population: {sales_like['Description'].isna().sum():,}")

PASS: source frame unchanged
PASS: qualifying population satisfies Quantity > 0 and UnitPrice > 0
PASS: product keys, grouped rows, total quantity, and invoice bounds reconcile
PASS: description audit reconciles and its join preserves product rows
PASS: population matches the previously documented workbook counts
Missing Description rows retained in population: 0


## 7. Main results and interpretation

The next cell summarizes the leaders for each measure directly from the ranked tables. The full top-20 tables above retain the remaining ranking detail.

In [7]:
quantity_leader = top_by_quantity.iloc[0]
invoice_leader = top_by_invoices.iloc[0]
print("Main results")
print("Highest total recorded quantity: "
      f"StockCode {quantity_leader['StockCode']} ("
      f"{quantity_leader['canonical_description'] if pd.notna(quantity_leader['canonical_description']) else 'description unresolved'}), "
      f"{quantity_leader['total_quantity']:,} units across "
      f"{quantity_leader['distinct_invoices']:,} distinct qualifying invoices.")
print("Appears in the most distinct qualifying invoices: "
      f"StockCode {invoice_leader['StockCode']} ("
      f"{invoice_leader['canonical_description'] if pd.notna(invoice_leader['canonical_description']) else 'description unresolved'}), "
      f"{invoice_leader['distinct_invoices']:,} invoices and "
      f"{invoice_leader['total_quantity']:,} recorded units.")
print("All assertions in Section 6 passed.")

Main results
Highest total recorded quantity: StockCode 23843 (paper craft , little birdie), 80,995 units across 1 distinct qualifying invoices.
Appears in the most distinct qualifying invoices: StockCode 85123A (white hanging heart t-light holder), 2,198 invoices and 37,660 recorded units.
All assertions in Section 6 passed.


### Limitations

- Total quantity is recorded positive-line volume under a provisional sign filter. It is not net demand, unmet demand, or proof of completed sales; returns are excluded, and stock availability is not observed. Pack sizes and bulk purchases can also make unit totals hard to compare.
- Distinct invoice count measures how broadly a product appears across qualifying invoices, not how many distinct customers bought it or how many independent demand events occurred. Invoice splitting and basket composition can affect the count.
- The workbook covers a finite historical window. These rankings describe only this observed period and population; they do not establish future demand or product performance outside it.
- Description normalization is a lightweight label audit. Case and whitespace differences are collapsed, unresolved top-count ties are left ambiguous, and normalized labels do not validate that a stock code represents a single stable product.